In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np
import pandas as pd
import pickle
import os
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

BASE = '/kaggle/input/competitions/multi-lingual-video-fragment-retrieval-challenge/video-rag'

In [ ]:
video_files = pd.read_csv(f'{BASE}/video_files.csv')
audio_files = pd.read_csv(f'{BASE}/audio_files.csv')
train_qa = pd.read_csv(f'{BASE}/train/train_qa.csv')
test = pd.read_csv(f'{BASE}/test/test.csv')

print("=== video_files.csv ===")
print(video_files.head())
print(f"Shape: {video_files.shape}\n")

print("=== audio_files.csv ===")
print(audio_files.head())
print(f"Shape: {audio_files.shape}\n")

print("=== train_qa.csv ===")
print(train_qa.head())
print(f"Shape: {train_qa.shape}")
print(f"Columns: {train_qa.columns.tolist()}\n")

print("=== test.csv ===")
print(test.head())
print(f"Shape: {test.shape}\n")

with open(f'{BASE}/transcripts.pkl', 'rb') as f:
    transcripts = pickle.load(f)

print("=== transcripts.pkl ===")
print(f"Type: {type(transcripts)}")

if isinstance(transcripts, dict):
    keys = list(transcripts.keys())
    print(f"Keys count: {len(keys)}")
    print(f"Sample key: {keys[0]}")
    sample = transcripts[keys[0]]
    print(f"Sample value type: {type(sample)}")
    print(f"Sample value: {sample}")
elif isinstance(transcripts, list):
    print(f"Length: {len(transcripts)}")
    print(f"First item: {transcripts[0]}")
else:
    print(transcripts)

# EDA

In [ ]:
import pandas as pd
import numpy as np
import pickle

BASE = '/kaggle/input/competitions/multi-lingual-video-fragment-retrieval-challenge/video-rag'

train = pd.read_csv(f'{BASE}/train/train_qa.csv')
test  = pd.read_csv(f'{BASE}/test/test.csv')

with open(f'{BASE}/transcripts.pkl', 'rb') as f:
    transcripts = pickle.load(f)

# ── 1. Длины фрагментов ──────────────────────────────────────────
train['duration'] = train['end'] - train['start']

print("=== Длины фрагментов (end - start), секунды ===")
print(train['duration'].describe(percentiles=[.1, .25, .5, .75, .9, .95]))

# ── 2. Сколько видео на один вопрос ─────────────────────────────
vids_per_q = train.groupby('question_id')['video_file'].nunique()
print("\n=== Уникальных видео на один question_id ===")
print(vids_per_q.describe())
print("Распределение:")
print(vids_per_q.value_counts().sort_index())

# ── 3. Топики ────────────────────────────────────────────────────
print("\n=== Топики в трейне ===")
print(train['topic'].value_counts())

# ── 4. Язык запросов в тесте ────────────────────────────────────
import re

def detect_lang(text):
    cyrillic = len(re.findall(r'[а-яА-ЯёЁ]', str(text)))
    latin    = len(re.findall(r'[a-zA-Z]', str(text)))
    if cyrillic > latin:
        return 'ru'
    elif latin > cyrillic:
        return 'en'
    return 'mixed'

test['lang'] = test['question'].apply(detect_lang)
print("\n=== Язык запросов в test.csv ===")
print(test['lang'].value_counts())

# ── 5. Транскрипции: длины сегментов ────────────────────────────
seg_durations = []
seg_texts_len = []
video_total_segs = []

for key, segs in transcripts.items():
    video_total_segs.append(len(segs))
    for s in segs:
        seg_durations.append(s['end'] - s['start'])
        seg_texts_len.append(len(s['text'].split()))

seg_dur = pd.Series(seg_durations)
seg_txt = pd.Series(seg_texts_len)
vid_segs = pd.Series(video_total_segs)

print("\n=== Длина одного Whisper-сегмента, секунды ===")
print(seg_dur.describe(percentiles=[.25, .5, .75, .9]))

print("\n=== Слов в одном сегменте ===")
print(seg_txt.describe(percentiles=[.25, .5, .75, .9]))

print("\n=== Сегментов на видео ===")
print(vid_segs.describe(percentiles=[.25, .5, .75, .9]))

# ── 6. Покрытие: все ли видео из трейна есть в транскрипциях ────
import re as _re

def extract_hash(path):
    m = _re.search(r'_([a-f0-9]+)[\.\w]*$', str(path))
    return m.group(1) if m else None

transcript_hashes = set(extract_hash(k) for k in transcripts.keys())
train_video_hashes = set(train['video_file'].apply(extract_hash).dropna())
test_coverage = train_video_hashes - transcript_hashes

print(f"\n=== Покрытие транскрипций ===")
print(f"Уникальных видео в трейне:      {len(train_video_hashes)}")
print(f"Ключей в transcripts.pkl:        {len(transcript_hashes)}")
print(f"Видео без транскрипций:          {len(test_coverage)}")
if test_coverage:
    print(f"Примеры пропущенных хэшей:   {list(test_coverage)[:5]}")

# ── 7. Пример 3 вопросов из теста ───────────────────────────────
print("\n=== Примеры тестовых запросов ===")
print(test.sample(6, random_state=42)[['query_id', 'question', 'lang']].to_string(index=False))

# bladskiy STEP 1

In [ ]:
!pip install faiss-cpu==1.13.2
!pip install nvidia-smi

In [ ]:
import pickle
import re
import numpy as np
import pandas as pd
import faiss
import torch
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

In [ ]:
BASE = '/kaggle/input/competitions/multi-lingual-video-fragment-retrieval-challenge/video-rag'
WORK = '/kaggle/working'

WINDOW      = 60.0
STEP        = 30.0
SKIP_HASHES = {'7d49c038'}
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Device: {DEVICE}")

In [ ]:
def extract_hash(path: str):
    m = re.search(r'_([a-f0-9]+)[\.\w]*$', str(path))
    return m.group(1) if m else None

def clean_text(text: str) -> str:
    return text.lower().strip()

def make_chunks(segments: list, window: float, step: float) -> list:
    if not segments:
        return []
    chunks = []
    total_end = segments[-1]['end']
    t_start   = segments[0]['start']
    while t_start < total_end:
        t_end       = t_start + window
        window_segs = [s for s in segments if s['end'] > t_start and s['start'] < t_end]
        if window_segs:
            text = ' '.join(clean_text(s['text']) for s in window_segs)
            chunks.append({
                'start': window_segs[0]['start'],
                'end':   window_segs[-1]['end'],
                'text':  text,
            })
        t_start += step
    return chunks

print("Загружаем транскрипции...")
with open(f'{BASE}/transcripts.pkl', 'rb') as f:
    transcripts = pickle.load(f)
print(f"Видео в pkl: {len(transcripts)}")

print("Строим чанки...")
all_chunks = []
for key, segments in tqdm(transcripts.items()):
    video_hash = extract_hash(key)
    if video_hash is None or video_hash in SKIP_HASHES:
        continue
    for ch in make_chunks(segments, WINDOW, STEP):
        ch['video_hash'] = video_hash
        all_chunks.append(ch)

print(f"Всего чанков: {len(all_chunks)}")

In [ ]:
print("Загружаем модель на GPU...")
model = SentenceTransformer('intfloat/multilingual-e5-large', device=DEVICE)

texts = [f"passage: {ch['text']}" for ch in all_chunks]

print(f"Энкодим {len(texts)} чанков на {DEVICE}...")
embeddings = model.encode(
    texts,
    batch_size=128,       
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)
embeddings = embeddings.astype('float32')
print(f"Embeddings shape: {embeddings.shape}")

print("Строим FAISS индекс...")
dim        = embeddings.shape[1]
index_flat = faiss.IndexFlatIP(dim)

if faiss.get_num_gpus() > 0:
    print(f"FAISS переносим на GPU ({faiss.get_num_gpus()} устройств)")
    res   = faiss.StandardGpuResources()
    index = faiss.index_cpu_to_gpu(res, 0, index_flat)
else:
    print("FAISS на CPU")
    index = index_flat

index.add(embeddings)
print(f"Индекс построен. Векторов: {index.ntotal}")

# Сохраняем — GPU индекс нужно конвертировать обратно в CPU перед сохранением
if faiss.get_num_gpus() > 0:
    index_to_save = faiss.index_gpu_to_cpu(index)
else:
    index_to_save = index

faiss.write_index(index_to_save, f'{WORK}/index_iter1.faiss')

with open(f'{WORK}/chunks_iter1.pkl', 'wb') as f:
    pickle.dump(all_chunks, f)

print(f"\nСохранено: {WORK}/index_iter1.faiss")
print(f"Сохранено: {WORK}/chunks_iter1.pkl")
print("Блок A завершён.")

In [ ]:
# ════════════════════════════════════════════════════════════════
# БЛОК B — Инференс на тестовых запросах (Итерация 1)
# ════════════════════════════════════════════════════════════════

TOP_K_RETRIEVE = 50   # сколько кандидатов берём из FAISS
TOP_K_RESULT   = 5    # сколько отдаём в ответ
MERGE_GAP      = 15.0 # секунд — чанки ближе этого расстояния объединяем

# ── Загрузка индекса и метаданных ───────────────────────────────
print("Загружаем индекс и чанки...")
index = faiss.read_index(f'{WORK}/index_iter1.faiss')

with open(f'{WORK}/chunks_iter1.pkl', 'rb') as f:
    all_chunks = pickle.load(f)

test = pd.read_csv(f'{BASE}/test/test.csv')
print(f"Индекс: {index.ntotal} векторов")
print(f"Тестовых запросов: {len(test)}")

# ── Загрузка video_files для маппинга hash → filename ───────────
video_files = pd.read_csv(f'{BASE}/video_files.csv')

# Строим словарь hash → имя файла без пути и расширения
# video_files.csv содержит: videos/video_abcd1234.mp4
# submission требует: video_abcd1234
hash_to_filename = {}
for path in video_files['video_path']:
    h = extract_hash(path)
    # убираем путь и расширение
    filename = re.sub(r'\.\w+$', '', path.split('/')[-1])
    if h:
        hash_to_filename[h] = filename

print(f"Маппинг видео: {len(hash_to_filename)} файлов")

# ── Постобработка кандидатов ─────────────────────────────────────
def merge_chunks(candidates: list, gap: float) -> list:
    """
    Объединяет перекрывающиеся или близкие чанки из одного видео.
    candidates: [{'video_hash', 'start', 'end', 'score'}, ...]
    Возвращает отсортированный по score список без дублей.
    """
    # Группируем по видео
    by_video = {}
    for c in candidates:
        vh = c['video_hash']
        if vh not in by_video:
            by_video[vh] = []
        by_video[vh].append(c)

    merged = []
    for vh, chks in by_video.items():
        # Сортируем по start внутри видео
        chks = sorted(chks, key=lambda x: x['start'])

        current = chks[0].copy()
        for nxt in chks[1:]:
            # Если следующий чанк перекрывается или рядом — объединяем
            if nxt['start'] <= current['end'] + gap:
                current['end']   = max(current['end'], nxt['end'])
                current['score'] = max(current['score'], nxt['score'])
            else:
                merged.append(current)
                current = nxt.copy()
        merged.append(current)

    # Сортируем итоговые фрагменты по score
    merged.sort(key=lambda x: x['score'], reverse=True)
    return merged


def retrieve(query: str, model, index, all_chunks: list,
             top_k_retrieve: int = 50, top_k_result: int = 5,
             merge_gap: float = 15.0) -> list:
    """
    Для одного запроса возвращает top_k_result фрагментов:
    [{'video_hash', 'start', 'end', 'score'}, ...]
    """
    # Энкодим запрос
    query_vec = model.encode(
        [f"query: {query.lower().strip()}"],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype('float32')

    # Поиск в FAISS
    scores, indices = index.search(query_vec, top_k_retrieve)
    scores  = scores[0]
    indices = indices[0]

    # Собираем кандидатов
    candidates = []
    for score, idx in zip(scores, indices):
        if idx == -1:
            continue
        ch = all_chunks[idx]
        candidates.append({
            'video_hash': ch['video_hash'],
            'start':      ch['start'],
            'end':        ch['end'],
            'score':      float(score),
        })

    # Merge соседних чанков
    merged = merge_chunks(candidates, gap=merge_gap)

    return merged[:top_k_result]


# ── Прогон по всем тестовым запросам ────────────────────────────
print(f"\nЗапускаем инференс на {len(test)} запросах...")
results = []

for _, row in tqdm(test.iterrows(), total=len(test)):
    query_id = row['query_id']
    question = str(row['question'])

    hits = retrieve(
        query        = question,
        model        = model,
        index        = index,
        all_chunks   = all_chunks,
        top_k_retrieve = TOP_K_RETRIEVE,
        top_k_result   = TOP_K_RESULT,
        merge_gap      = MERGE_GAP,
    )

    results.append({'query_id': query_id, 'hits': hits})

print(f"Инференс завершён. Примеров: {len(results)}")

# Проверка первого результата
print("\nПример результата:")
r = results[0]
print(f"query_id: {r['query_id']}")
for i, h in enumerate(r['hits']):
    fname = hash_to_filename.get(h['video_hash'], h['video_hash'])
    print(f"  #{i+1} {fname}  [{h['start']:.1f}s – {h['end']:.1f}s]  score={h['score']:.4f}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# БЛОК C — Формирование submission.csv
# ════════════════════════════════════════════════════════════════

def build_submission(results: list, hash_to_filename: dict,
                     top_k: int = 5) -> pd.DataFrame:
    rows = []
    for r in results:
        row = {'query_id': r['query_id']}
        hits = r['hits']

        for rank in range(1, top_k + 1):
            if rank <= len(hits):
                h     = hits[rank - 1]
                fname = hash_to_filename.get(h['video_hash'], h['video_hash'])
                row[f'video_file_{rank}'] = fname
                row[f'start_{rank}']      = round(h['start'], 1)
                row[f'end_{rank}']        = round(h['end'],   1)
            else:
                # Если меньше 5 результатов — заполняем пустыми
                row[f'video_file_{rank}'] = ''
                row[f'start_{rank}']      = 0.0
                row[f'end_{rank}']        = 0.0

        rows.append(row)

    cols = ['query_id']
    for rank in range(1, top_k + 1):
        cols += [f'video_file_{rank}', f'start_{rank}', f'end_{rank}']

    return pd.DataFrame(rows, columns=cols)


submission = build_submission(results, hash_to_filename)

print(f"Submission shape: {submission.shape}")
print(submission.head(3).to_string())

# Сохраняем
submission.to_csv(f'{WORK}/submission_iter1.csv', index=False)
print(f"\nСохранено: {WORK}/submission_iter1.csv")

# Быстрая валидация
assert submission['query_id'].nunique() == len(test), "Не все query_id в сабмите"
assert not submission['video_file_1'].isna().any(),   "Есть NaN в video_file_1"
print("Валидация пройдена. Можно сабмитить.")

In [ ]:
# ════════════════════════════════════════════════════════════════
# Валидация на train — понимаем где теряем SR и VR
# ════════════════════════════════════════════════════════════════

train = pd.read_csv(f'{BASE}/train/train_qa.csv')

# Строим ground truth: question_id → список (video_hash, start, end)
gt = {}
for _, row in train.iterrows():
    qid = row['question_id']
    vh  = extract_hash(row['video_file'])
    if qid not in gt:
        gt[qid] = []
    gt[qid].append({'video_hash': vh, 'start': row['start'], 'end': row['end']})

def iou(pred_start, pred_end, gt_start, gt_end) -> float:
    inter = max(0, min(pred_end, gt_end) - max(pred_start, gt_start))
    union = max(pred_end, gt_end) - min(pred_start, gt_start)
    return inter / union if union > 0 else 0.0

def evaluate(gt: dict, results: list, hash_to_filename: dict,
             ks=(1, 3, 5), iou_thresh=0.5):
    """
    Считаем SR@K и VR@K на подмножестве вопросов из трейна.
    """
    # Берём только те query_id которые есть в трейне
    train_qids = set(gt.keys())

    sr = {k: [] for k in ks}
    vr = {k: [] for k in ks}

    for r in results:
        qid = r['query_id']
        if qid not in train_qids:
            continue

        gt_items = gt[qid]
        hits     = r['hits']

        for k in ks:
            top_hits = hits[:k]

            # VR@K — правильное видео есть среди топ-K (без учёта таймкодов)
            pred_hashes = {h['video_hash'] for h in top_hits}
            gt_hashes   = {g['video_hash'] for g in gt_items}
            vr_hit      = int(bool(pred_hashes & gt_hashes))
            vr[k].append(vr_hit)

            # SR@K — хотя бы один хит с IoU >= 0.5 и правильным видео
            sr_hit = 0
            for h in top_hits:
                for g in gt_items:
                    if h['video_hash'] == g['video_hash']:
                        if iou(h['start'], h['end'], g['start'], g['end']) >= iou_thresh:
                            sr_hit = 1
                            break
                if sr_hit:
                    break
            sr[k].append(sr_hit)

    metrics = {}
    for k in ks:
        metrics[f'SR@{k}'] = np.mean(sr[k])
        metrics[f'VR@{k}'] = np.mean(vr[k])

    avg_sr = np.mean([metrics[f'SR@{k}'] for k in ks])
    avg_vr = np.mean([metrics[f'VR@{k}'] for k in ks])
    metrics['AvgSR']      = avg_sr
    metrics['AvgVR']      = avg_vr
    metrics['FinalScore'] = (avg_sr + avg_vr) / 2

    return metrics


# Прогон на трейне — используем те же question_id что в тесте (overlap)
# Но тест и трейн не пересекаются по query_id, поэтому делаем отдельный прогон
# по подвыборке трейна как отдельные запросы

print("Прогон retrieve на подвыборке трейна (первые 200 вопросов)...")

# Берём уникальные вопросы из трейна
train_questions = (
    train[['question_id', 'question_en']]
    .drop_duplicates('question_id')
    # .head(200)
)

train_results = []
for _, row in tqdm(train_questions.iterrows(), total=len(train_questions)):
    hits = retrieve(
        query          = row['question_en'],
        model          = model,
        index          = index,
        all_chunks     = all_chunks,
        top_k_retrieve = TOP_K_RETRIEVE,
        top_k_result   = TOP_K_RESULT,
        merge_gap      = MERGE_GAP,
    )
    train_results.append({'query_id': row['question_id'], 'hits': hits})

metrics = evaluate(gt, train_results, hash_to_filename)

print("\n=== Метрики на трейне===")
for k in (1, 3, 5):
    print(f"  SR@{k} = {metrics[f'SR@{k}']:.4f}    VR@{k} = {metrics[f'VR@{k}']:.4f}")
print(f"\n  AvgSR      = {metrics['AvgSR']:.4f}")
print(f"  AvgVR      = {metrics['AvgVR']:.4f}")
print(f"  FinalScore = {metrics['FinalScore']:.4f}")